In [ ]:
%%writefile przyklad.txt
1. 2 3 5
2. 1 3 4 5 7
3. 6
4. 2 7
5. 7
6. 2
7. 6

In [ ]:
from zad1 import (
    generuj_digraf_gnp,
    konwertuj_digraf,
    wczytaj_digraf,
    wizualizuj_digraf,
    wypisz_liste_sasiedztwa,
    wypisz_macierz,
)

n, luki = wczytaj_digraf("przyklad.txt")
A, B, L = konwertuj_digraf(n, luki)

print("LISTA SĄSIEDZTWA:")
wypisz_liste_sasiedztwa(L)

print("\nMACIERZ SĄSIEDZTWA:")
wypisz_macierz(A)

print("\nMACIERZ INCYDENCJI:")
wypisz_macierz(B)

wizualizuj_digraf(n, luki, tytul="Graf")

assert generuj_digraf_gnp(0, 0.5) == []
assert generuj_digraf_gnp(1, 1.0) == []
assert generuj_digraf_gnp(5, 0.0) == []
assert len(generuj_digraf_gnp(5, 1.0)) == 20

In [ ]:
from zad1 import generuj_digraf_gnp, wczytaj_digraf, wizualizuj_digraf
from zad2 import czy_silnie_spojny, kolory_skladowych, kosaraju, wypisz_skladowe

n, luki = wczytaj_digraf("przyklad.txt")
skl = kosaraju(n, luki)
print("Digraf:", czy_silnie_spojny(n, luki))
wypisz_skladowe(skl)

wizualizuj_digraf(
    n, luki,
    kolory=kolory_skladowych(n, skl),
    tytul=f"Przykład - {len(skl)} SCC",
)

n, p = 8, 0.25
luki = generuj_digraf_gnp(n, p, seed=None)
skladowe = kosaraju(n, luki)
print(f"\nLosowy digraf G({n}, {p}) - liczba składowych: {len(skladowe)}")
wypisz_skladowe(skladowe)
wizualizuj_digraf(
    n, luki,
    kolory=kolory_skladowych(n, skladowe),
    tytul=f"Kosaraju na G({n}, {p}) - {len(skladowe)} SCC",
)

assert kosaraju(0, []) == [] and not czy_silnie_spojny(0, [])
assert kosaraju(1, []) == [[0]] and czy_silnie_spojny(1, [])
assert kosaraju(1, [(0, 0)]) == [[0]]
assert kosaraju(4, [(0, 1), (1, 2), (2, 3)]) == [[0], [1], [2], [3]]  # Graf acykliczny
assert kosaraju(4, [(0, 1), (1, 2), (2, 3), (3, 0)]) == [[0, 1, 2, 3]]  # Domknięty cykl z wyżej
assert kosaraju(6, [(0, 1), (1, 2), (2, 0), (2, 3), (3, 4), (4, 5), (5, 3)]) == [
    [0, 1, 2], [3, 4, 5]
]  # dwa cykle połączone jednym łukiem (słabo spójny, niesilnie spójny)
assert len(kosaraju(1500, [(i, i + 1) for i in range(1499)])) == 1500  # głęboka ścieżka

In [ ]:
%%writefile przyklad_wazony.txt
7 14
1 2 6
1 3 3
1 5 -1
2 1 10
2 3 -5
2 4 -4
2 5 4
2 7 4
3 6 2
4 2 5
4 7 9
5 7 -4
6 2 9
7 6 4

In [ ]:
import math

from zad1 import wizualizuj_digraf

from zad3 import (
    CyklUjemnyError,
    bellman_ford,
    generuj_silnie_spojny_wazony,
    wczytaj_digraf_wazony,
    wypisz_sciezki,
)

n, wagi = wczytaj_digraf_wazony("przyklad_wazony.txt")
d1, p1 = bellman_ford(n, wagi, start=0)
d2, p2 = bellman_ford(n, wagi, start=1)

print("Najkrótsze ścieżki z wierzchołka 1:")
wypisz_sciezki(0, d1, p1)
wizualizuj_digraf(n, list(wagi.keys()), wagi=wagi, tytul="Digraf ważony")

n_los = 6
wagi_los = generuj_silnie_spojny_wazony(
    n_los, p=0.45, min_w=-5, max_w=10, bez_cyklu_ujemnego=True, seed=None
)
d_los, p_los = bellman_ford(n_los, wagi_los, start=0)
print("\nLosowy silnie spójny digraf:")
wypisz_sciezki(0, d_los, p_los)
wizualizuj_digraf(n_los, list(wagi_los.keys()), wagi=wagi_los, tytul="Losowy silnie spójny digraf ważony")

assert bellman_ford(2, {(0, 1): 3, (1, 0): -3}, start=0)[0] == [0, 3]

assert bellman_ford(4, {(0, 1): 2, (2, 3): -1, (3, 2): -1}, start=0)[0] == [0, 2, math.inf, math.inf]

try:
    bellman_ford(1, {(0, 0): -1}, start=0)
    assert False, "Powinien wykryć ujemną pętlę własną"
except CyklUjemnyError:
    pass

assert bellman_ford(1, {(0, 0): 4}, start=0)[0] == [0]

In [ ]:
import math

from zad4 import (
    johnson,
    weryfikuj_johnsona,
    wizualizuj_przewazenie,
    wypisz_macierz_odleglosci,
)

from zad3 import CyklUjemnyError, wczytaj_digraf_wazony

n, wagi = wczytaj_digraf_wazony("przyklad_wazony.txt")
D, h, wagi_prim = johnson(n, wagi, zwroc_szczegoly=True)

assert all(w >= 0 for w in wagi_prim.values())
assert weryfikuj_johnsona(n, wagi, D)

print("Macierz odległości:")
wypisz_macierz_odleglosci(D)
wizualizuj_przewazenie(n, wagi, wagi_prim, h)

assert johnson(0, {}) == []                                   # digraf pusty
assert johnson(1, {}) == [[0]]                                # pojedynczy wierzchołek
assert johnson(1, {(0, 0): 5}) == [[0]]                       # dodatnia pętla własna
assert johnson(2, {(0, 1): 4, (1, 0): -4}) == [[0, 4], [-4, 0]]  # cykl wagi 0
assert johnson(3, {(0, 1): -2}) == [                          # niespójny z wierzchołkiem izolowanym
    [0, -2, math.inf],
    [math.inf, 0, math.inf],
    [math.inf, math.inf, 0],
]

try:
    johnson(4, {(0, 1): 1, (2, 3): -1, (3, 2): -1})
    assert False, "Cykl ujemny"
except CyklUjemnyError:
    pass

try:
    johnson(2, {(0, 1): 2, (1, 1): -1})
    assert False, "Pętla własna"
except CyklUjemnyError:
    pass
